<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #1f1c2c 0%, #928dab 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.25);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎭 Fish Audio Drama 3 · int4</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Kaggle GPU T4 x2 Edition · <strong>experimental</strong></h3>
  <p style='color: #e5e5e5; margin: 0;'>35B MoE (3B active) expressive TTS | 83 languages | voice cloning | multi-speaker scenes | int4 experts split over both T4s</p>
</div>

---

<div align="center">
  <img src="https://img.shields.io/badge/Kaggle-GPU%20T4%20x2-20BEFF?style=for-the-badge&logo=kaggle&logoColor=white" />
  <img src="https://img.shields.io/badge/Model-fishaudio%2Fdrama--3--int4-orange?style=for-the-badge&logo=huggingface&logoColor=white" />
  <img src="https://img.shields.io/badge/Status-experimental-yellow?style=for-the-badge" />
</div>

---

### What is Drama 3?

**Drama 3** is Fish Audio's open-weights speech model for performances rather than read-outs: you direct the delivery in plain words inside `[brackets]` (`[whispering]`, `[shaky breath]`, `[cracking up despite herself]`), write several characters with `<|speaker:N|>`, and clone any voice from a short reference clip with its transcript.

### How does a 35B model fit on two 16 GB T4s?

The official int4 checkpoint (`fishaudio/drama-3-int4`, 22 GB) is built for GPTQ-Marlin, which needs an Ampere GPU (sm80+). The T4 is Turing (sm75), so this notebook runs it differently:

- **Engine:** Fish Audio's own pure-PyTorch Drama 3 code (`fishaudio/fish-speech`, pinned commit), in FP16 with an FP32 residual stream (T4 has no BF16)
- **Routed experts stay int4** on the GPU (16.6 GB). A small Triton kernel dequantizes only the 8 experts each frame uses, inside the matmul (~9x faster than unpacking them in PyTorch)
- **Dense int4 projections** are unpacked once to FP16 at load time; the 1 GB token embedding stays in CPU RAM; the output head keeps only the 4,097 rows Drama 3 can sample
- **Split:** layers 0-28 on GPU 0; layers 29-39, the codebook tower and the codec on GPU 1

### Measured on Kaggle T4 x2

| | |
|---|---|
| Load time | ~2 min (after a ~6 min download) |
| Speed | about **3-4x slower than real time** (10 s of audio ≈ 30-40 s) |
| VRAM | ~13.4 GiB on GPU 0, ~11.5 GiB on GPU 1 |

### ⚠️ Before you start
1. **Accelerator:** GPU T4 x2 · **Internet:** on
2. Optional: add a Kaggle secret named `HF_TOKEN` (Add-ons → Secrets) to avoid Hugging Face's anonymous rate limit during the 22 GB download
3. **License:** Fish Audio Community License: free for research and non-commercial use, and for commercial use under USD 500k annual revenue. Offering Drama 3 as a hosted TTS/cloning service needs a separate license. Never clone a real person's voice without their consent.

---
## ⚙️ Step 1: Setup + model download

Clones Fish Audio's `fish-speech` (pinned commit), installs its packages and downloads `fishaudio/drama-3-int4` (~22 GB) to `/kaggle/tmp`. An interrupted download resumes on its own.

In [ ]:
# Step 1: pinned fish-speech + dependencies + int4 weights (~22 GB)
import os, subprocess, signal
# Free the GPUs if an earlier run's engine is still alive
subprocess.run("pkill -f run_drama3.py", shell=True)
# Step 1 logic: pinned fish-speech + deps + int4 weights. Shared by the notebook and the test kernel.
import os
import subprocess
import sys
import time

FISH_SPEECH_COMMIT = "541566932abbaebcb0a40e562a1f846738383f38"  # fishaudio/fish-speech, 14 Oct 2026
MODEL_REPO = "fishaudio/drama-3-int4"
ROOT = "/kaggle/tmp" if os.path.isdir("/kaggle") else os.path.abspath(".")
os.makedirs(ROOT, exist_ok=True)
REPO_DIR = os.path.join(ROOT, "fish-speech")
CKPT_DIR = os.path.join(ROOT, "drama-3-int4")


def sh(cmd, **kw):
    print("$", cmd if isinstance(cmd, str) else " ".join(cmd), flush=True)
    subprocess.run(cmd, check=True, shell=isinstance(cmd, str), **kw)


def setup():
    t0 = time.time()
    try:  # optional: a HF_TOKEN Kaggle secret lifts Hugging Face's anonymous rate limit
        from kaggle_secrets import UserSecretsClient
        os.environ.setdefault("HF_TOKEN", UserSecretsClient().get_secret("HF_TOKEN"))
    except Exception:
        pass
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
    os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
    if not os.path.isdir(os.path.join(REPO_DIR, ".git")):
        sh(["git", "clone", "-q", "https://github.com/fishaudio/fish-speech", REPO_DIR])
    sh(["git", "-C", REPO_DIR, "checkout", "-q", FISH_SPEECH_COMMIT])
    pkgs = [
        "transformers==5.19.0", "tokenizers==0.23.2", "hydra-core>=1.3.2", "omegaconf", "natsort", "einops>=0.7.0",
        "einx[torch]==0.2.2", "loguru", "pyrootutils", "resampy", "librosa", "pydub", "ormsgpack", "cachetools",
        "kui>=1.6.0", "uvicorn", "tiktoken", "safetensors", "soundfile", "opencc-python-reimplemented==0.1.7",
        "huggingface_hub>=0.34", "hf_xet", "gradio>=5.0",
        # runtime deps of descript-audiotools / descript-audio-codec (installed below without their pins)
        "argbind", "pyloudnorm", "julius", "rich", "flatten_dict", "markdown2", "randomname", "pystoi",
        "torch_stoi", "importlib_resources", "baize", "ffmpy", "tensorboard", "matplotlib",
    ]
    sh([sys.executable, "-m", "pip", "install", "-q", *pkgs])
    # descript-audio-codec drags an ancient protobuf pin through audiotools: install them without deps.
    sh([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "descript-audio-codec", "descript-audiotools"])
    sh([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", REPO_DIR])
    from huggingface_hub import snapshot_download
    print(f"⬇️  Downloading {MODEL_REPO} (~22 GB)...", flush=True)
    for attempt in range(1, 9):
        try:
            snapshot_download(
                MODEL_REPO, local_dir=CKPT_DIR,
                allow_patterns=["*.safetensors", "model.safetensors.index.json", "config.json", "codec.pth",
                                "tokenizer.json", "tokenizer_config.json", "special_tokens_map.json",
                                "chat_template.jinja", "LICENSE"],
                max_workers=8,
            )
            break
        except Exception as e:  # HF answers 429 to bursts of anonymous downloads: wait, then resume
            if attempt == 8:
                raise
            wait = 45 * attempt
            print(f"⚠️  Download interrupted ({str(e)[:120]}). Resuming in {wait}s (attempt {attempt}/8)...", flush=True)
            time.sleep(wait)
    print(f"✅ Setup done in {time.time() - t0:.0f}s", flush=True)


setup()

---
## 🚀 Step 2: Start the engine + Gradio UI

Writes two files into the fish-speech folder, then starts the engine in the background:
- `drama3_int4_t4.py`: the T4 patch (int4 experts + Triton kernel, GPU split, trimmed output head)
- `run_drama3.py`: the engine and the Gradio interface

Loading takes about 2 minutes, then the cell prints the public **Gradio** link. Every generation logs its length, time and VRAM in `/kaggle/working/drama3.log`. Re-run this cell to restart the engine.

In [ ]:
# Step 2: write the T4 patch + app, then launch them
import os, subprocess, sys, time, re

PATCH_CODE = r'''"""Run fishaudio/drama-3-int4 with fish-speech's pure PyTorch Drama 3 engine on GPUs without
GPTQ-Marlin (Turing T4 / sm75, Volta...).

Import this module BEFORE building the model. It patches fish-speech in place:

* Routed MoE experts stay packed in int4 on the GPU (GPTQ layout: qweight int32 [K/8, N],
  fp16 scales [K/128, N], symmetric, zero point 8). Only the experts a token is routed to are
  dequantized, on the fly, right before their matmul. 16.6 GB of experts instead of 64 GB.
* The int4 dense projections (full-attention q/k/v/o, linear-attention in_proj_qkv / in_proj_z /
  out_proj) are dequantized once at load time into ordinary fp16 nn.Linear weights (~2.5 GB).
* The backbone is split over several GPUs with an adjustable share per GPU, so the last GPU keeps
  room for the codebook tower and the codec (fish-speech's default puts the whole backbone on
  every GPU but the last one).

Env:
  DRAMA3_GPU_SHARES   comma list of backbone shares per GPU, default "0.68,0.32"
"""
import os
import re
from collections import defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
from loguru import logger

from fish_speech.models.text2semantic.fish_omni import backbone as fo_backbone
from fish_speech.models.text2semantic.fish_omni import model as fo_model
from fish_speech.models.text2semantic.fish_omni import moe as fo_moe

GROUP = 128
GPTQ_SYM_ZEROS = -2004318072  # int32 0x88888888: eight packed zero points of 8
GATHERED_MAX_ROWS = fo_moe.GATHERED_EXPERT_MAX_ROWS

_SHIFTS: dict[torch.device, torch.Tensor] = {}


def _shifts(device: torch.device) -> torch.Tensor:
    s = _SHIFTS.get(device)
    if s is None:
        s = torch.arange(0, 32, 4, dtype=torch.int32, device=device).view(8, 1)
        _SHIFTS[device] = s
    return s


def dequant_int4(qweight: torch.Tensor, scales: torch.Tensor, dtype: torch.dtype) -> torch.Tensor:
    """GPTQ sym int4 [..., K/8, N] int32 + scales [..., K/128, N] -> weight [..., K, N] (in x out)."""
    *lead, k8, n = qweight.shape
    k = k8 * 8
    q = (qweight.unsqueeze(-2) >> _shifts(qweight.device)) & 0xF  # [..., K/8, 8, N], row k = 8*i + j
    q = q.reshape(*lead, k // GROUP, GROUP, n).to(dtype) - 8
    w = q * scales.to(dtype).unsqueeze(-2)
    return w.reshape(*lead, k, n)



# ---- Triton W4A16 GEMV for the routed experts (decode). Works on sm75; falls back to torch. ----
try:
    import triton
    import triton.language as tl

    @triton.jit
    def _w4_expert_gemv(x_ptr, qw_ptr, sc_ptr, ids_ptr, out_ptr, K: tl.constexpr, N: tl.constexpr,
                        BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr):
        m = tl.program_id(0)
        nb = tl.program_id(1)
        e = tl.load(ids_ptr + m).to(tl.int64)
        offs_n = nb * BLOCK_N + tl.arange(0, BLOCK_N)
        mask_n = offs_n < N
        qw_base = qw_ptr + e * (K // 8) * N
        sc_base = sc_ptr + e * (K // 128) * N
        offs_i = tl.arange(0, BLOCK_K // 8)
        acc = tl.zeros((BLOCK_N,), dtype=tl.float32)
        for k0 in range(0, K, BLOCK_K):
            qw = tl.load(qw_base + (k0 // 8 + offs_i)[:, None] * N + offs_n[None, :], mask=mask_n[None, :], other=0)
            part = tl.zeros((BLOCK_N,), dtype=tl.float32)
            for j in tl.static_range(8):  # nibble j of word i holds input row k0 + 8 * i + j
                qj = ((qw >> (4 * j)) & 0xF).to(tl.float32) - 8.0
                xj = tl.load(x_ptr + m * K + k0 + offs_i * 8 + j).to(tl.float32)
                part += tl.sum(xj[:, None] * qj, axis=0)
            s = tl.load(sc_base + (k0 // 128) * N + offs_n, mask=mask_n, other=0.0).to(tl.float32)
            acc += part * s
        tl.store(out_ptr + m * N + offs_n, acc.to(out_ptr.dtype.element_ty), mask=mask_n)

    def w4_expert_gemv(x, qweight, scales, ids, block_n=64):
        """x [M, K] (row m pairs with expert ids[m]), qweight [E, K/8, N] -> [M, N]."""
        M, K = x.shape
        N = qweight.shape[-1]
        out = torch.empty((M, N), dtype=x.dtype, device=x.device)
        grid = (M, triton.cdiv(N, block_n))
        with torch.cuda.device(x.device):
            _w4_expert_gemv[grid](x.contiguous(), qweight, scales, ids.to(torch.int32).contiguous(), out, K, N,
                                  BLOCK_N=block_n, BLOCK_K=128, num_warps=4)
        return out

    HAVE_TRITON = True
except Exception as _e:  # pragma: no cover
    HAVE_TRITON = False
    logger.warning(f"Triton unavailable ({_e}); int4 experts use the torch path")

USE_TRITON = HAVE_TRITON and os.environ.get("DRAMA3_TRITON", "1") != "0"
TRITON_MAX_PAIRS = 16384  # 2,048 prompt rows x 8 experts


class Int4MoEExperts(nn.Module):
    """Drop-in for fish-speech MoEExperts holding GPTQ int4 experts.

    w13_qweight [E, H/8, 2I] (gate columns, then up), w2_qweight [E, I/8, H]; fp16 group scales.
    """

    def __init__(self, num_experts, hidden, intermediate, *, fp8=False, dtype=None, device=None):
        super().__init__()
        self.num_experts = num_experts
        self.hidden = hidden
        self.intermediate = intermediate
        self.fp8 = False
        sdt = torch.float16
        self.register_buffer("w13_qweight", torch.empty((num_experts, hidden // 8, 2 * intermediate), dtype=torch.int32, device=device))
        self.register_buffer("w13_scales", torch.empty((num_experts, hidden // GROUP, 2 * intermediate), dtype=sdt, device=device))
        self.register_buffer("w2_qweight", torch.empty((num_experts, intermediate // 8, hidden), dtype=torch.int32, device=device))
        self.register_buffer("w2_scales", torch.empty((num_experts, intermediate // GROUP, hidden), dtype=sdt, device=device))

    def forward(self, x, topk_ids, topk_weights):
        # The Triton GEMV also serves prefill (one program row per token-expert pair): far faster on
        # T4 than looping over the experts and unpacking each one in PyTorch.
        if USE_TRITON and x.is_cuda and topk_ids.numel() <= TRITON_MAX_PAIRS:
            return self.forward_gathered(x, topk_ids, topk_weights)
        if topk_ids.numel() <= GATHERED_MAX_ROWS:
            return self.forward_gathered(x, topk_ids, topk_weights)
        return self.forward_grouped(x, topk_ids, topk_weights)

    def forward_gathered(self, x, topk_ids, topk_weights):
        global USE_TRITON
        if USE_TRITON and x.is_cuda:
            try:
                return self.forward_triton(x, topk_ids, topk_weights)
            except Exception as e:
                USE_TRITON = False
                logger.warning(f"Triton int4 kernel failed ({e!r:.300}); switching to the torch path")
        return self.forward_gathered_torch(x, topk_ids, topk_weights)

    def forward_triton(self, x, topk_ids, topk_weights):
        tokens, top_k = topk_ids.shape
        ids = topk_ids.reshape(-1)
        rows = x.repeat_interleave(top_k, dim=0)
        gate, up = w4_expert_gemv(rows, self.w13_qweight, self.w13_scales, ids).chunk(2, dim=-1)
        hidden = F.silu(gate) * up
        out = w4_expert_gemv(hidden, self.w2_qweight, self.w2_scales, ids)
        out = out.float() * topk_weights.reshape(-1, 1).float()
        return out.view(tokens, top_k, self.hidden).sum(dim=1).to(x.dtype)

    def forward_gathered_torch(self, x, topk_ids, topk_weights):
        tokens, top_k = topk_ids.shape
        ids = topk_ids.reshape(-1)
        w13 = dequant_int4(self.w13_qweight[ids], self.w13_scales[ids], x.dtype)  # [M, H, 2I]
        rows = x.repeat_interleave(top_k, dim=0).unsqueeze(1)  # [M, 1, H]
        gate, up = torch.bmm(rows, w13).squeeze(1).chunk(2, dim=-1)
        del w13
        hidden = (F.silu(gate) * up).unsqueeze(1)  # [M, 1, I]
        w2 = dequant_int4(self.w2_qweight[ids], self.w2_scales[ids], x.dtype)  # [M, I, H]
        out = torch.bmm(hidden, w2).squeeze(1)
        out = out.float() * topk_weights.reshape(-1, 1).float()
        return out.view(tokens, top_k, self.hidden).sum(dim=1).to(x.dtype)

    def forward_grouped(self, x, topk_ids, topk_weights):
        tokens, top_k = topk_ids.shape
        flat = topk_ids.reshape(-1)
        order = torch.argsort(flat, stable=True)
        token_rows = order // top_k
        pair_weights = topk_weights.reshape(-1)[order]
        counts = torch.bincount(flat, minlength=self.num_experts).tolist()
        out = torch.zeros(tokens, self.hidden, dtype=torch.float32, device=x.device)
        start = 0
        for expert, count in enumerate(counts):
            if count == 0:
                continue
            rows = token_rows[start:start + count]
            weights = pair_weights[start:start + count]
            start += count
            w13 = dequant_int4(self.w13_qweight[expert], self.w13_scales[expert], x.dtype)
            gate, up = (x[rows] @ w13).chunk(2, dim=-1)
            w2 = dequant_int4(self.w2_qweight[expert], self.w2_scales[expert], x.dtype)
            expert_out = (F.silu(gate) * up) @ w2
            out.index_add_(0, rows, expert_out.float() * weights[:, None].float())
        return out.to(x.dtype)


EXPERT_RE = re.compile(r"model\.layers\.(\d+)\.mlp\.experts\.(\d+)\.(gate_proj|up_proj|down_proj)\.(qweight|scales)")
DENSE_FIELDS = ("qweight", "qzeros", "scales", "g_idx")


@torch.no_grad()
def load_int4_backbone_weights(model, weights):
    targets = dict(model.named_parameters())
    targets.update(model.named_buffers())
    expected = set(targets)
    loaded: set[str] = set()
    expert_parts: dict[str, int] = defaultdict(int)
    dense: dict[str, dict[str, torch.Tensor]] = defaultdict(dict)

    def put(name, tensor):
        if name not in targets:
            raise ValueError(f"checkpoint tensor has no parameter {name!r}")
        target = targets[name]
        if target.shape != tensor.shape:
            raise ValueError(f"{name}: checkpoint {tuple(tensor.shape)} != model {tuple(target.shape)}")
        if target.dtype == torch.float16 and tensor.dtype != torch.float16:
            tensor = fo_moe.clamp_to_float16(tensor.float(), name)
        target.copy_(tensor)
        loaded.add(name)

    for ck_name, tensor in weights:
        name = fo_backbone.backbone_name(ck_name)
        m = EXPERT_RE.fullmatch(name)
        if m:
            layer, expert, proj, kind = m.groups()
            prefix = f"model.layers.{layer}.mlp.experts"
            if proj == "down_proj":
                full = f"{prefix}.w2_{kind}"
                dst = targets[full][int(expert)]
            else:
                full = f"{prefix}.w13_{kind}"
                cols = targets[full].shape[-1] // 2
                sl = slice(0, cols) if proj == "gate_proj" else slice(cols, 2 * cols)
                dst = targets[full][int(expert)][:, sl]
            if dst.shape != tensor.shape:
                raise ValueError(f"{ck_name}: {tuple(tensor.shape)} != {tuple(dst.shape)}")
            dst.copy_(tensor)
            expert_parts[full] += 1
            continue
        base, _, field = name.rpartition(".")
        if field in DENSE_FIELDS:
            dense[base][field] = tensor
            if len(dense[base]) == len(DENSE_FIELDS):
                parts = dense.pop(base)
                if not bool((parts["qzeros"] == GPTQ_SYM_ZEROS).all()):
                    raise ValueError(f"{base}: only symmetric GPTQ (zero point 8) is supported")
                target = targets[base + ".weight"]
                w = dequant_int4(parts["qweight"].to(target.device), parts["scales"].to(target.device), torch.float32)
                put(base + ".weight", w.t().contiguous())
            continue
        put(name, tensor)

    if dense:
        raise ValueError(f"{len(dense)} int4 linears are incomplete, e.g. {sorted(dense)[:4]}")
    for full, count in expert_parts.items():
        experts = targets[full].shape[0]
        need = experts * (1 if ".w2_" in full else 2)
        if count != need:
            raise ValueError(f"{full}: {count} expert parts loaded, expected {need}")
        loaded.add(full)
    missing = expected - loaded
    if missing:
        raise ValueError(f"{len(missing)} backbone tensors were not loaded, e.g. {sorted(missing)[:8]}")
    logger.info(f"Drama 3 int4: loaded {len(loaded)} backbone tensors (int4 experts kept packed)")


def materialize_with_shares(self, devices):
    """FishOmniBackbone.materialize with a configurable share of the backbone per device."""
    devices = [torch.device(d) for d in devices]
    shares = [float(s) for s in os.environ.get("DRAMA3_GPU_SHARES", "0.68,0.32").split(",")]
    if len(shares) != len(devices):
        shares = [1.0 / len(devices)] * len(devices)
    total_share = sum(shares)
    bounds, acc = [], 0.0
    for s in shares:
        acc += s / total_share
        bounds.append(acc)
    layer_bytes = [fo_backbone.module_bytes(layer) for layer in self.model.layers]
    head = fo_backbone.module_bytes(self.model.embed_tokens)
    tail = fo_backbone.module_bytes(self.lm_head) + fo_backbone.module_bytes(self.model.norm)
    # The 1 GB token embedding is a lookup table: keep it in CPU RAM, ship rows to the GPU.
    total = sum(layer_bytes) + tail
    head = 0
    self.model.embed_tokens.to_empty(device="cpu")
    self.model.norm.to_empty(device=devices[-1])
    self.lm_head.to_empty(device=devices[-1])
    placement = defaultdict(list)
    cumulative = head
    for index, (layer, size) in enumerate(zip(self.model.layers, layer_bytes)):
        middle = (cumulative + size / 2) / total
        slot = next((i for i, b in enumerate(bounds) if middle <= b), len(devices) - 1)
        layer.to_empty(device=devices[slot])
        placement[str(devices[slot])].append(index)
        cumulative += size
    self.inverse_frequencies.clear()
    logger.info("Drama 3 int4 placement: " + ", ".join(
        f"{d}: layers {v[0]}-{v[-1]}" for d, v in placement.items()))


def embed_on_cpu(self, input_ids):
    table = self.model.embed_tokens
    first = self.model.layers[0].input_layernorm.weight.device
    return table(input_ids.to(table.weight.device)).to(first, non_blocking=True)


def trim_lm_head(model):
    """Drama 3 only ever samples the main token over eos + the semantic band (4,097 of 252,288 rows):
    keep those rows of the 1 GB lm_head and scatter their logits back into a full-vocabulary row."""
    bb, cfg = model.backbone, model.config
    rows = torch.tensor(sorted({cfg.eos_token_id, *range(cfg.semantic_start_token_id, cfg.semantic_end_token_id + 1)}))
    head = bb.lm_head
    dev = head.weight.device
    bb.lm_rows = rows.to(dev)
    bb.lm_vocab = head.weight.shape[0]
    head.weight = nn.Parameter(head.weight.data[bb.lm_rows].clone(), requires_grad=False)
    torch.cuda.empty_cache()
    logger.info(f"Drama 3 int4: lm_head trimmed to {len(rows)} sampled rows on {dev}")


def compute_logits_trimmed(self, hidden):
    head = self.lm_head
    small = F.linear(hidden.to(head.weight.device), head.weight)
    if not hasattr(self, "lm_rows"):
        return small
    full = small.new_full((*small.shape[:-1], self.lm_vocab), float("-inf"))
    full[..., self.lm_rows] = small
    return full


def resolve_devices_all(device: str):
    """Backbone over every GPU (shares above); tower + codec on the last one."""
    specs = [s.strip() for s in device.split(",") if s.strip()]
    if specs == ["cuda"]:
        specs = [f"cuda:{i}" for i in range(torch.cuda.device_count())]
    return specs, specs[-1]


def apply():
    from fish_speech.models.text2semantic.fish_omni.config import FishOmniConfig

    original_from_dict = FishOmniConfig.from_dict.__func__

    def from_dict(cls, data):
        q = data.get("quantization_config")
        if q is not None and q.get("quant_method") == "drama3-int4":
            data = {k: v for k, v in data.items() if k != "quantization_config"}
        return original_from_dict(cls, data)

    FishOmniConfig.from_dict = classmethod(from_dict)
    fo_moe.MoEExperts = Int4MoEExperts
    fo_model.load_backbone_weights = load_int4_backbone_weights
    fo_backbone.FishOmniBackbone.materialize = materialize_with_shares
    fo_backbone.FishOmniBackbone.embed = embed_on_cpu
    fo_backbone.FishOmniBackbone.compute_logits = compute_logits_trimmed
    original_from_pretrained = fo_model.FishOmniModel.from_pretrained.__func__

    def from_pretrained(cls, *args, **kwargs):
        model = original_from_pretrained(cls, *args, **kwargs)
        trim_lm_head(model)
        return model

    fo_model.FishOmniModel.from_pretrained = classmethod(from_pretrained)
    # FishOmniModel.from_pretrained decides "fp8 experts" from the checkpoint's quantization
    # config; our experts ignore that flag, but keep the dequantize-to-dtype branch quiet.
    os.environ.setdefault("FISH_DRAMA3_FP8_EXPERTS", "0")
    import fish_speech.models.text2semantic.inference as fs_inf
    fs_inf.resolve_drama3_devices = resolve_devices_all
    try:
        import tools.server.model_manager as mm
        mm.resolve_drama3_devices = resolve_devices_all
    except Exception:
        pass
    logger.info("Drama 3 int4 patch applied (Turing-friendly int4 experts, multi-GPU split)")


apply()
'''

RUN_CODE = r'''"""Fish Audio Drama 3 (int4) on 2x T4 - engine + Gradio UI.

python run_drama3.py --ckpt /kaggle/tmp/drama-3-int4 [--selftest OUT_DIR] [--port 7860]
Run from the fish-speech repo root (pinned commit). Imports drama3_int4_t4 first.
"""
import argparse
import io
import json
import os
import sys
import threading
import time

sys.path.insert(0, os.getcwd())
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("FISH_DRAMA3_CUDA_GRAPH", "0")  # graphs need a single-GPU model

import numpy as np
import soundfile as sf
import torch

import drama3_int4_t4  # noqa: F401  (patches fish-speech)
from fish_speech.utils.schema import ServeReferenceAudio, ServeTTSRequest
from tools.server.model_manager import ModelManager

REF_URL = "https://huggingface.co/datasets/zhaochenyang20/seed-tts-eval-mini/resolve/main/en/prompt-wavs/common_voice_en_10119832.wav"
REF_TEXT = "We asked over twenty different people, and they all said it was his."


BENCH = {}


def gpu_report():
    out = []
    for i in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(i)
        out.append(f"GPU{i}: {(total - free) / 2**30:.1f}/{total / 2**30:.1f} GiB used, peak alloc {torch.cuda.max_memory_allocated(i) / 2**30:.1f} GiB")
    return " | ".join(out)


_orig_warm_up = ModelManager.warm_up


def _warm_up(self, engine):
    print(f"📦 Weights loaded · {gpu_report()}", flush=True)
    _orig_warm_up(self, engine)


ModelManager.warm_up = _warm_up


class Engine:
    def __init__(self, ckpt):
        t0 = time.time()
        self.manager = ModelManager(
            mode="tts",
            device="cuda",
            half=True,
            compile=False,
            llama_checkpoint_path=ckpt,
            decoder_checkpoint_path=os.path.join(ckpt, "codec.pth"),
            decoder_config_name="modded_dac_vq",
        )
        self.engine = self.manager.tts_inference_engine
        self.lock = threading.Lock()
        print(f"✅ Engine ready in {time.time() - t0:.0f}s · {gpu_report()}", flush=True)

    def tts(self, text, ref_audio_bytes=None, ref_text="", temperature=0.7, top_p=0.7,
            max_new_tokens=0, seed=None, single_chunk=False):
        refs = []
        if ref_audio_bytes:
            refs.append(ServeReferenceAudio(audio=ref_audio_bytes, text=ref_text or ""))
        req = ServeTTSRequest(
            text=text, references=refs, format="wav", temperature=float(temperature),
            top_p=float(top_p), max_new_tokens=int(max_new_tokens) if max_new_tokens else None,
            seed=int(seed) if seed not in (None, "", -1) else None, single_chunk=bool(single_chunk),
        )
        with self.lock:
            t0 = time.time()
            audio = None
            for result in self.engine.inference(req):
                if result.code == "error":
                    raise RuntimeError(str(result.error))
                if result.code == "final" and isinstance(result.audio, tuple):
                    audio = result.audio
            took = time.time() - t0
            torch.cuda.empty_cache()
        if audio is None:
            raise RuntimeError("no audio generated")
        sr, wav = audio
        dur = len(wav) / sr
        print(f"🎙️ {dur:.1f}s of audio in {took:.1f}s (RTF {took / max(dur, 1e-6):.2f}) · {gpu_report()}", flush=True)
        return sr, wav, took


def bench_experts():
    """Check the Triton int4 kernel against the torch path on every layer, then time both."""
    import gc
    backbone = next((o for o in gc.get_objects() if type(o).__name__ == "FishOmniBackbone"), None)
    if backbone is None:
        return {}
    res = {"max_rel_err": {}}
    torch.manual_seed(0)
    if drama3_int4_t4.USE_TRITON:
        for li, layer in enumerate(backbone.model.layers):
            ex = layer.mlp.experts
            dev = ex.w13_qweight.device
            worst = 0.0
            for _ in range(4):
                x = (torch.randn(1, ex.hidden, device=dev) * 2).half()
                ids = torch.randperm(ex.num_experts, device=dev)[:8][None]
                w = torch.softmax(torch.randn(1, 8, device=dev), -1).half()
                a = ex.forward_triton(x, ids, w).float()
                b = ex.forward_gathered_torch(x, ids, w).float()
                worst = max(worst, float((a - b).norm() / b.norm().clamp(min=1e-6)))
            res["max_rel_err"][f"L{li}@{dev}"] = round(worst, 5)
        bad = {k: v for k, v in res["max_rel_err"].items() if v > 0.01}
        if bad:
            drama3_int4_t4.USE_TRITON = False
            res["triton_disabled"] = bad
    ex = backbone.model.layers[0].mlp.experts
    dev = ex.w13_qweight.device
    x = torch.randn(1, ex.hidden, device=dev, dtype=torch.float16)
    ids = torch.randint(0, ex.num_experts, (1, 8), device=dev)
    w = torch.softmax(torch.randn(1, 8, device=dev), -1).half()
    for name, fn in (("torch", ex.forward_gathered_torch), ("triton", ex.forward_triton)):
        if name == "triton" and not drama3_int4_t4.USE_TRITON:
            continue
        fn(x, ids, w)
        torch.cuda.synchronize(dev)
        t0 = time.time()
        for _ in range(40):
            fn(x, ids, w)
        torch.cuda.synchronize(dev)
        res[f"{name}_ms_per_frame"] = round((time.time() - t0) * 1000, 2)
    print("⏱️ experts check:", json.dumps(res), flush=True)
    return res


def selftest(engine, out_dir):
    import urllib.request
    os.makedirs(out_dir, exist_ok=True)
    bench = BENCH
    ref = urllib.request.urlopen(REF_URL, timeout=60).read()
    cases = [
        ("plain", "Hello, how are you? The weather is lovely today, so let's take a long walk by the river.", None),
        ("tags", "[barely holding it together, forcing a smile] I'm fine. [shaky breath] Really. [small laugh] [whispering] Go on without me.", None),
        ("clone", "[sighs] Fine. If we miss it, it's on you. [laughs] Well, maybe that's fair.", ref),
        ("french", "Bonjour à tous ! [excited] Aujourd'hui, on teste Drama 3 en local, sur deux petites cartes T4.", None),
    ]
    if os.environ.get("DRAMA3_AB"):
        cases = [(f"{c[0]}_{mode}_s{seed}", c[1], c[2], mode, seed) for seed in (1, 2, 3, 4)
                 for mode in ("triton", "torch") for c in cases if c[0] == "clone"]
    else:
        cases = [(c[0], c[1], c[2], "triton", 42) for c in cases]
    report = []
    for name, text, r, mode, seed in cases:
        drama3_int4_t4.USE_TRITON = mode == "triton" and drama3_int4_t4.HAVE_TRITON
        try:
            sr, wav, took = engine.tts(text, r, REF_TEXT if r else "", seed=seed,
                                       max_new_tokens=500 if os.environ.get("DRAMA3_AB") else 0)
            sf.write(os.path.join(out_dir, f"{name}.wav"), wav, sr)
            report.append({"case": name, "ok": True, "seconds": round(took, 1), "audio_s": round(len(wav) / sr, 2),
                           "peak": float(np.abs(wav).max()), "rms": float(np.sqrt(np.mean(wav ** 2)))})
        except Exception as e:  # keep going: we want every case's verdict
            import traceback
            traceback.print_exc()
            report.append({"case": name, "ok": False, "error": repr(e)[:500]})
        print(json.dumps(report[-1]), flush=True)
    report.append({"gpu": gpu_report(), "bench": bench})
    with open(os.path.join(out_dir, "selftest.json"), "w") as f:
        json.dump(report, f, indent=1)


def build_ui(engine):
    import gradio as gr

    examples = [
        ["[barely holding it together, forcing a smile] I'm fine. [shaky breath] Really. [small laugh] [whispering] Go on without me."],
        ["[through gritted teeth, keeping his voice down] I asked you to do one thing tonight. [slow, dangerously quiet] Just one."],
        ["[straight-faced, trying very hard to stay serious] No, I love it, it really suits you. [cracking up despite herself] I'm sorry, it's just a very brave haircut."],
        ["<|speaker:0|>[nervous] Did you hear that? <|speaker:1|>[whispering] Stay quiet. Don't move."],
        ["Bonjour à tous ! [excited] Aujourd'hui, on fait tourner Drama 3 en local, sur deux petites cartes T4."],
    ]

    def run(text, ref_audio, ref_text, temperature, top_p, max_new, seed, single_chunk):
        if not text or not text.strip():
            raise gr.Error("Write a line to perform first.")
        ref_bytes = None
        if ref_audio:
            if not (ref_text or "").strip():
                raise gr.Error("Voice cloning needs the exact transcript of the reference clip.")
            with open(ref_audio, "rb") as f:
                ref_bytes = f.read()
        sr, wav, took = engine.tts(text, ref_bytes, ref_text, temperature, top_p, max_new, seed, single_chunk)
        dur = len(wav) / sr
        return (sr, wav), f"{dur:.1f} s of audio in {took:.1f} s · RTF {took / max(dur, 1e-6):.2f}"

    with gr.Blocks(title="Fish Audio Drama 3 · int4 on T4 x2") as demo:
        gr.Markdown("# 🎭 Fish Audio Drama 3 · int4 on Kaggle T4 x2\n"
                    "Direct the performance with `[brackets]`, write the line in any of 83 languages, "
                    "use `<|speaker:N|>` for several characters, and add a reference clip to clone a voice.")
        with gr.Row():
            with gr.Column(scale=3):
                text = gr.Textbox(label="Script", lines=6, value=examples[0][0])
                with gr.Accordion("Voice cloning (optional)", open=False):
                    ref_audio = gr.Audio(label="Reference clip (5-30 s, one speaker)", type="filepath")
                    ref_text = gr.Textbox(label="Exact transcript of the reference clip")
                with gr.Accordion("Sampling", open=False):
                    temperature = gr.Slider(0.1, 1.0, 0.7, step=0.05, label="Temperature")
                    top_p = gr.Slider(0.1, 1.0, 0.7, step=0.05, label="Top-p")
                    max_new = gr.Slider(0, 2048, 0, step=64, label="Max frames per chunk (0 = model default)")
                    seed = gr.Number(-1, label="Seed (-1 = random)", precision=0)
                    single_chunk = gr.Checkbox(False, label="Generate the whole text in one pass")
                go = gr.Button("🎬 Perform", variant="primary")
            with gr.Column(scale=2):
                audio = gr.Audio(label="Take", type="numpy")
                info = gr.Markdown()
        gr.Examples(examples, inputs=[text])
        go.click(run, [text, ref_audio, ref_text, temperature, top_p, max_new, seed, single_chunk], [audio, info])
    return demo


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--ckpt", required=True)
    p.add_argument("--selftest")
    p.add_argument("--port", type=int, default=7860)
    p.add_argument("--share", action="store_true")
    a = p.parse_args()
    print(f"torch {torch.__version__} · {torch.cuda.device_count()} GPU(s): "
          + ", ".join(torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())), flush=True)
    engine = Engine(a.ckpt)
    BENCH.update(bench_experts())  # also validates the Triton kernel before any request
    if a.selftest:
        selftest(engine, a.selftest)
        return
    demo = build_ui(engine)
    demo.queue(max_size=8).launch(server_name="0.0.0.0", server_port=a.port, share=a.share, show_error=True)


if __name__ == "__main__":
    main()
'''

with open(os.path.join(REPO_DIR, "drama3_int4_t4.py"), "w") as f:
    f.write(PATCH_CODE)
with open(os.path.join(REPO_DIR, "run_drama3.py"), "w") as f:
    f.write(RUN_CODE)

subprocess.run("pkill -f run_drama3.py", shell=True)
time.sleep(2)
LOG = "/kaggle/working/drama3.log" if os.path.isdir("/kaggle/working") else "drama3.log"
log = open(LOG, "w")
engine = subprocess.Popen([sys.executable, "-u", "run_drama3.py", "--ckpt", CKPT_DIR, "--share"],
                          cwd=REPO_DIR, stdout=log, stderr=subprocess.STDOUT)
print("⏳ Loading Drama 3 (about 2 minutes)...")
seen, local_at = 0, None
while True:
    time.sleep(3)
    text = open(LOG, errors="replace").read()
    for line in text[seen:].splitlines():
        if any(k in line for k in ("placement", "📦", "✅", "⏱️", "Running on", "Error", "error")):
            print(line[:300])
    seen = len(text)
    if "gradio.live" in text:
        print("\n🎬 Ready. Open the gradio.live link above.")
        break
    if "Running on local URL" in text:
        local_at = local_at or time.time()
        if time.time() - local_at > 90:
            print("\n⚠️ Gradio's share link did not come up; the UI only runs at the local URL above.")
            break
    if engine.poll() is not None:
        print(text[-4000:])
        raise RuntimeError("The engine stopped; its log is printed above")

# "Save & Run All" (batch) sessions have nobody to click: take one test through the UI's API, then stop.
if os.environ.get("KAGGLE_KERNEL_RUN_TYPE") == "Batch":
    from gradio_client import Client
    import shutil
    client = Client("http://127.0.0.1:7860/", verbose=False)
    t0 = time.time()
    wav, info = client.predict("[warmly] Hello from two little T4 GPUs. [small laugh] Drama 3 runs here.",
                               None, "", 0.7, 0.7, 0, 42, False, api_name="/run")
    shutil.copy(wav, "/kaggle/working/batch_test.wav")
    print(f"🧪 Batch test: {info} (round trip {time.time() - t0:.0f}s)")
    engine.terminate()

---
## 🎬 Writing for Drama 3

- **Direct the line in brackets, in plain words:** `[sarcastic, slow]`, `[through gritted teeth]`, `[small laugh]`, `[whispering]`. Several directions can follow each other.
- **Several characters:** start each turn with `<|speaker:0|>`, `<|speaker:1|>`...
- **Voice cloning:** a clean 5-30 s clip of one speaker, plus its exact transcript.
- **Any of 83 languages**, including French: the directions can stay in English.
- Sampling is random: if a take derails (runs on, goes silent, drops the end), press Perform again or lower the temperature.
- Long texts are split into chunks automatically; *Generate the whole text in one pass* keeps the delivery continuous but needs more VRAM.

**Troubleshooting**
- `CUDA out of memory`: shorten the text or the reference clip, untick the one-pass option, then retry.
- Download stuck on HTTP 429: add the `HF_TOKEN` secret and re-run Step 1 (it resumes).

<p align="center" style="color:#6b7280; font-size:12px; margin-top:8px;">
  Engine: <a href="https://github.com/fishaudio/fish-speech">fishaudio/fish-speech</a> · Weights: <a href="https://huggingface.co/fishaudio/drama-3-int4">fishaudio/drama-3-int4</a> · Fish Audio Community License
</p>